# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Mã chính nằm ở `lab/vlm.py`; notebook gọi các hàm trong đó.

## 0. Môi trường (Colab T4)

In [ ]:
# Runtime → Change runtime type → T4 GPU
import os
REPO = '/content/K4-Track-4-Day10-VisionLanguageModel'
if not os.path.exists(REPO):
    !git clone -q https://github.com/kascenite/K4-Track-4-Day10-VisionLanguageModel {REPO}
%cd {REPO}
!git pull -q
!nvidia-smi -L
!pip -q install -r requirements-colab.txt pytest

In [ ]:
# Drive: work/ (dự đoán, adapter, run.json) nằm trên Drive để không mất khi session bị ngắt
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/lab_day10_work'
os.makedirs(DRIVE, exist_ok=True)
if not os.path.islink('work'):
    !rm -rf work && ln -s {DRIVE} work

In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

## 1. Dữ liệu: tải và chia split

In [ ]:
if not os.path.exists('data/splits/meta.json'):
    !python scripts/prepare_data.py --kaggle
!python -m pytest -q 2>&1 | tail -2

In [ ]:
import importlib, json, collections
import lab.vlm as V; importlib.reload(V)
from PIL import ImageDraw
splits = {s: V.load_split(s) for s in ('train', 'val', 'test')}
sess = {s: {e['session'] for e in rows} for s, rows in splits.items()}
assert not (sess['train'] & sess['val'] or sess['train'] & sess['test'] or sess['val'] & sess['test'])
for s, rows in splits.items():
    c = collections.Counter(o['label'] for e in rows for o in e['objects'])
    print(s, len(rows), 'ảnh', dict(c), len(sess[s]), 'phiên')
print(json.load(open('data/splits/meta.json')))
COLORS = {'pothole': 'red', 'crack': 'yellow', 'manhole': 'cyan'}
def draw(e, boxes, scale=1000):
    img = V.load_image(e); d = ImageDraw.Draw(img); W, H = img.size
    for b in boxes:
        x1, y1, x2, y2 = b['bbox_2d']
        d.rectangle([x1 / scale * W, y1 / scale * H, x2 / scale * W, y2 / scale * H], outline=COLORS.get(b['label'], 'white'), width=2)
    return img

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
import matplotlib.pyplot as plt
fig, axs = plt.subplots(2, 3, figsize=(15, 6))
for ax, e in zip(axs.flat, splits['train'][::240]):
    ans = V.objects_to_answer(e['objects'])
    ax.imshow(draw(e, json.loads(ans))); ax.set_title(e['image_id'][:30], fontsize=8); ax.axis('off')
plt.tight_layout(); os.makedirs('report/figures', exist_ok=True); plt.savefig('report/figures/train_samples.png', dpi=80)
e = splits['train'][0]; print(V.objects_to_answer(e['objects'])); print(V.PROMPT_NORM)

## 3. E0: zero-shot

In [ ]:
import torch
model, processor = V.load_model()
print([n for n, _ in model.named_modules() if n.endswith('layers.0.self_attn.q_proj') or n.endswith('blocks.0.attn.qkv') or n.endswith('merger')])
print(sum(p.numel() for p in model.parameters()) / 1e9, 'B (4-bit params đếm theo gói)')

In [ ]:
V.predict(model, processor, splits['val'][:40], 'work/e0_val40.jsonl')
!python scripts/score.py --pred work/e0_val40.jsonl --split val --name E0val --out work/e0_val40.json

In [ ]:
V.predict(model, processor, splits['test'], 'work/e0.jsonl')
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json
for r in V.read_jsonl('work/e0.jsonl')[:5]: print(repr(r['raw'][:300]))

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
e = splits['train'][0]
ex = V.encode_example(processor, V.load_image(e), V.PROMPT_NORM, V.objects_to_answer(e['objects']))
img_tok = processor.tokenizer.convert_tokens_to_ids('<|image_pad|>')
print('keys', list(ex), 'image tokens:', int((ex['input_ids'] == img_tok).sum()), 'grid', ex['image_grid_thw'].tolist())
print('labels decode:', repr(processor.tokenizer.decode(ex['input_ids'][ex['labels'] != -100])))
b = V.collate([ex, V.encode_example(processor, V.load_image(splits['train'][1]), V.PROMPT_NORM, V.objects_to_answer(splits['train'][1]['objects']))], processor.tokenizer.pad_token_id)
print({k: tuple(v.shape) for k, v in b.items()})

## 5. Setup LoRA / đóng băng cho E1, E2, E3
Kiểm tra số module khớp (252 / 348) và số tham số huấn luyện.

In [ ]:
import gc
def free():
    global model, pm
    for n in ('model', 'pm'):
        if n in globals(): del globals()[n]
    gc.collect(); torch.cuda.empty_cache()

SETUPS = {'e1': dict(train_projector=False, vit_lora=False),
          'e2': dict(train_projector=True, vit_lora=False),
          'e3': dict(train_projector=True, vit_lora=True)}

def run_setup(key, train_entries=None, coord='norm1000', r=16, **setup):
    run_dir = f'work/runs/{key}'
    if not os.path.exists(f'{run_dir}/run.json'):
        free()
        m, proc = V.load_model()
        pm, n_matched, n_train = V.apply_setup(m, r=r, alpha=2 * r, **setup)
        run = V.train(pm, proc, train_entries or splits['train'], run_dir, setup['train_projector'], coord=coord)
        print({k: v for k, v in run.items() if k != 'losses'})
        del m, pm; gc.collect(); torch.cuda.empty_cache()
    return run_dir

def eval_setup(key, run_dir, name, coord='norm1000'):
    free()
    pm, proc = V.load_trained(run_dir)
    V.predict(pm, proc, splits['test'], f'work/{key}.jsonl', prompt=V.prompt_for(coord))
    del pm; gc.collect(); torch.cuda.empty_cache()
    os.system(f'python scripts/score.py --pred work/{key}.jsonl --name {name} --coord {coord} '
              f'--out report/evidence/{key}.json {V.meta_args(run_dir)} 2>&1')
    print(open(f'report/evidence/{key}.json').read()[:600])
    !cp report/evidence/*.json work/

In [ ]:
free()
m, _ = V.load_model()
for k, s in SETUPS.items():
    pm, n_matched, n_train = V.apply_setup(m, **s)
    print(k, n_matched, f'{n_train:,}')
    m = pm.unload() if hasattr(pm, 'unload') else pm.base_model.model
    del pm
del m; gc.collect(); torch.cuda.empty_cache()

## 6. Vòng huấn luyện
`run_setup`: nạp model mới → `apply_setup` → `train` (lưu adapter + projector + run.json lên Drive).

## 7. Suy luận và chấm E1–E3
`eval_setup`: nạp lại từ đĩa (adapter + projector) → suy luận test → chấm.

In [ ]:
run_dir = run_setup('e1', **SETUPS['e1']); eval_setup('e1', run_dir, 'E1')

In [ ]:
run_dir = run_setup('e2', **SETUPS['e2']); eval_setup('e2', run_dir, 'E2')

In [ ]:
run_dir = run_setup('e3', **SETUPS['e3']); eval_setup('e3', run_dir, 'E3')

## 8. E4: ablation `data_size` (10% vs 100% ảnh train) trên setup tốt nhất

In [ ]:
res = {k: json.load(open(f'report/evidence/{k}.json'))['micro']['f1'] for k in SETUPS}
best = max(res, key=res.get); print(res, 'best =', best)
import random
sub = random.Random(42).sample(splits['train'], int(0.1 * len(splits['train'])))
run_dir = run_setup('e4_data_size_0.1', train_entries=sub, **SETUPS[best])
eval_setup('e4_data_size_0.1', run_dir, f'E4 data_size=0.1 ({best})')
!cp report/evidence/{best}.json report/evidence/e4_data_size_1.0.json
!python - <<'X'
import json; p='report/evidence/e4_data_size_1.0.json'; d=json.load(open(p)); d['name']='E4 data_size=1.0 ('+d['name']+')'; json.dump(d, open(p,'w'), ensure_ascii=False, indent=2)
X
!cp report/evidence/*.json work/

## 9. E5 (tuỳ chọn): GRPO với reward IoU

In [ ]:
# xem lab/grpo.py (nếu có thời gian)

## Tổng hợp

In [ ]:
!python scripts/check_submission.py
for k in sorted(os.listdir('report/evidence')):
    if k.endswith('.json'):
        d = json.load(open(f'report/evidence/{k}')); m = d['micro']
        print(k, f"F1={m['f1']:.3f}", [round(x, 3) for x in d['f1_ci95']], f"FP/img={d['fp_per_image']:.2f}", d['meta'])